In [ ]:
import pandas as pd
import numpy as np
import re
from functools import reduce
import seaborn as sns
import missingno as msno
%matplotlib inline

import matplotlib.pyplot as plt
from scipy.stats import norm

In [ ]:
# Set the figure size - handy for larger output
from matplotlib import pyplot as plt
plt.rcParams["figure.figsize"] = [10, 6]
# Set up with a higher resolution screen (useful on Mac)
%config InlineBackend.figure_format = 'retina'

# import data

In [ ]:
data_concat = pd.read_csv('D:\RESEARCH IN AFRICA\Manuscripts/research strend in Africa wos/data/studies_with_african_all_based_on_affil_only_and_group.csv')
data_concat

In [ ]:
data_prep_all = pd.read_csv('D:\RESEARCH IN AFRICA\Manuscripts/comprarison of databases/data/preprocessing_wos/concat_and_preprocessing_wos.csv')
data_prep_all = data_prep_all.drop_duplicates('UT (Unique WOS ID)')
data_prep_all

# top 5

In [ ]:
top5 = ['South Africa','Egypt','Nigeria','Tunisia','Algeria']

In [ ]:
others_notTop5 = ['Eritrea', 'Ghana', 'Mauritania', 'Mauritius', 'Congo Republic', 'Equatorial Guinea', 'South Sudan', 
                 'Uganda', "Cote d'Ivoire", 'Kenya', 'Lesotho', 'Niger', 'Central African Republic', 'Sao Tome and Principe', 
                 'Togo','Tanzania', 'Mozambique', 'Zambia', 'Guinea-Bissau', 'Benin', 'Guinea', 'Senegal', 'Cameroon', 
                 'Zimbabwe','DR Congo', 'Madagascar', 'Seychelles', 'Liberia', 'Burundi', 'Namibia', 'Rwanda', 'Malawi', 
                 'Mali','Eswatini', 'Cabo Verde', 'Somalia', 'Djibouti', 'Libya', 'Morocco', 'Gambia', 'Angola', 'Comoros',
                 'Ethiopia', 'Sierra Leone', 'Botswana', 'Chad', 'Sudan', 'Burkina Faso', 'Gabon']

In [ ]:
groups_keys = ['top5', 'others_notTop5']

In [ ]:
groups_values = [top5, others_notTop5]

In [ ]:
# using zip()
# to convert lists to dictionary
groups_keys_values = dict(zip(groups_keys, groups_values))

In [ ]:
for group_key in groups_keys:
    data_concat[group_key] = data_concat[groups_keys_values[group_key]].count(axis=1)

In [ ]:
data_concat = data_concat.replace(0, np.nan)
data_concat

## quels sont les pays ayant contribués et leurs contributions

In [ ]:
contries_df = data_concat.count().sort_values(ascending=False).to_frame().rename(columns={0:'count'})
contries_df

In [ ]:
contries_df.head(60)

In [ ]:

contries_df['percentage'] = contries_df['count']/len(data_concat)*100

#save data
contries_df.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/research strend in Africa wos/data/contries_participation.csv')

contries_df

In [ ]:
contries_df.head(60)

# evolution of research in Africa

1. Évolution de la recherche en Afrique de 1975 à 2021

Évolution de  la quantité de recherche pour :
- Global
- ou par région
- Par pays 

In [ ]:
year_count = data_concat['year'].fillna('NA').value_counts().to_frame().reset_index().rename(columns={'index':'year','year':'count'})
#save data
year_count.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/research strend in Africa wos/data/year_count.csv')

year_count

In [ ]:
year_count_dropna= year_count[year_count.year != 'NA']

In [ ]:
from sklearn.linear_model import Ridge

lr = Ridge()
lr.fit(year_count_dropna[['year']], year_count_dropna['count'])

# Ridge(alpha=1.0, copy_X=True, fit_intercept=True, max_iter=None,
#    normalize=False, random_state=None, solver='auto', tol=0.001)

plt.bar(year_count_dropna['year'], year_count_dropna['count'])

plt.plot(year_count_dropna['year'], lr.coef_*year_count_dropna['year']+lr.intercept_, color='orange')

## Build bar plot graph
plt.title("Research trend in Africa")
plt.xlabel("Publication years")
plt.ylabel("Number of publication")

# Save files in pdf and eps format
# plt.savefig("../paper/figures/Fig.1 Research trend in Africa.jpeg", dpi=300)
# plt.savefig("../paper/figures/Fig.1 Research trend in Africa.png", dpi=300)

In [ ]:
data_concat_year = data_concat.drop(['PubMed_ID_wos','doi','source_title'], axis=1).set_index('wos_ID')

data_concat_year

In [ ]:
data_concat_year_table = data_concat_year.groupby(['year']).count()
#save data
data_concat_year_table.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/research strend in Africa wos/data/year_count_countries.csv')

data_concat_year_table

In [ ]:
data_concat_year_group_table = data_concat_year.groupby(['year_group']).count()

#save data
data_concat_year_group_table.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/research strend in Africa wos/data/year_group_count_countries.csv')

data_concat_year_group_table

# Types de documents

Ici il faudrait définir et comparer l'évolution de la proportion des types de documents par région :
- par année et par pays
- ou par région
- ou global 

In [ ]:
# document contries

In [ ]:
data_concat.shape

In [ ]:
data_prep_all.shape

In [ ]:
data_prep_all_doc_types = data_prep_all[['UT (Unique WOS ID)','Document Type']].rename(columns={'UT (Unique WOS ID)':'wos_ID'})
data_prep_all_doc_types['wos_ID'] = data_prep_all_doc_types['wos_ID'].str.lower()
data_prep_all_doc_types = data_prep_all_doc_types.set_index('wos_ID')
data_prep_all_doc_types

In [ ]:
data_prep_all_doc_types2 = data_prep_all_doc_types['Document Type'].str.split('; ', expand=True)
data_prep_all_doc_types2

In [ ]:
data_prep_all_doc_types3 = pd.DataFrame()
for col in data_prep_all_doc_types2.columns:
    data = data_prep_all_doc_types2[[col]].dropna().rename(columns={col:0})
    
    data_prep_all_doc_types3 = pd.concat([data_prep_all_doc_types3, data], axis=0)
data_prep_all_doc_types3 = data_prep_all_doc_types3.rename(columns={0:'Document Type'}).reset_index().drop_duplicates()
data_prep_all_doc_types3

In [ ]:
data_concat_doc_types = data_concat.merge(data_prep_all_doc_types3, on = 'wos_ID', how='left')
data_concat_doc_types

In [ ]:
data_concat_doc_types_table = data_concat_doc_types.groupby(['Document Type']).count()

#save data
data_concat_doc_types_table.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/research strend in Africa wos/data/doc_types_table_count_countries.csv')

data_concat_doc_types_table

In [ ]:
data_concat_doc_types_yes_year_table = data_concat_doc_types.groupby(['year','Document Type']).count().reset_index()
#save data
data_concat_doc_types_yes_year_table.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/research strend in Africa wos/data/doc_types_yes_table_count_countries_year.csv')
data_concat_doc_types_yes_year_table

In [ ]:
data_concat_doc_types_yes_year_grp_table = data_concat_doc_types.groupby(['year_group','Document Type']).count().reset_index()
#save data
data_concat_doc_types_yes_year_grp_table.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/research strend in Africa wos/data/doc_types_yes_table_count_countries_year_group.csv')
data_concat_doc_types_yes_year_grp_table

In [ ]:
# document year

In [ ]:
data_concat_doc_types2 = data_concat_doc_types[['year','year_group','Document Type']].set_index(['year_group','year'])
data_concat_doc_types2

In [ ]:
# get dummies to have contries on columns
from functools import reduce
dfs = [pd.get_dummies(data_concat_doc_types2[col]) for col in data_concat_doc_types2]
d = reduce(lambda x,y : x.add(y, fill_value=0), dfs)

# data cleaning
cols = [c for c in d.columns if c[-1:] != '.']
df=d[cols]
cols = [c for c in df.columns.to_list() if len(c) != 1]
df=df[cols]

dfs= df.replace(0.0, np.nan).reset_index()
#save data
dfs.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/research strend in Africa wos/data/doc_types_table_count_years.csv')

dfs

In [ ]:
data_concat_doc_types_year_table = dfs.groupby(['year']).count()
#save data
data_concat_doc_types_year_table.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/research strend in Africa wos/data/doc_types_table_count_years.csv')

data_concat_doc_types_year_table

In [ ]:
data_concat_doc_types_year_g_table = dfs.groupby(['year_group']).count()
#save data
data_concat_doc_types_year_g_table.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/research strend in Africa wos/data/doc_types_table_count_years_group.csv')
data_concat_doc_types_year_g_table

# Langues

Proposition and evolution of publications language according to countries, région, Global 

In [ ]:
# language contries

In [ ]:
data_prep_all_Language = data_prep_all[['UT (Unique WOS ID)','Language']].rename(columns={'UT (Unique WOS ID)':'wos_ID'})
data_prep_all_Language['wos_ID'] = data_prep_all_Language['wos_ID'].str.lower()
data_prep_all_Language = data_prep_all_Language.set_index('wos_ID')
data_prep_all_Language

In [ ]:
data_prep_all_Language2 = data_prep_all_Language['Language'].str.split('; ', expand=True)
data_prep_all_Language2

In [ ]:
data_prep_all_Language3 = pd.DataFrame()
for col in data_prep_all_Language2.columns:
    data = data_prep_all_Language2[[col]].dropna().rename(columns={col:0})
    
    data_prep_all_Language3 = pd.concat([data_prep_all_Language3, data], axis=0)
data_prep_all_Language3 = data_prep_all_Language3.rename(columns={0:'Language'}).reset_index().drop_duplicates()
data_prep_all_Language3

In [ ]:
data_concat_Language = data_concat.merge(data_prep_all_Language3, on = 'wos_ID', how='left')
data_concat_Language

In [ ]:
data_concat_Language_table = data_concat_Language.groupby(['Language']).count()

#save data
data_concat_Language_table.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/research strend in Africa wos/data/langues_table_count_countries.csv')

data_concat_Language_table

In [ ]:
data_concat_Language_year_table = data_concat_Language.groupby(['year','Language']).count().reset_index()
#save data
data_concat_Language_year_table.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/research strend in Africa wos/data/languages_yes_table_count_countries_year.csv')
data_concat_Language_year_table

In [ ]:
data_concat_Language_year_grp_table = data_concat_Language.groupby(['year_group','Language']).count().reset_index()
#save data
data_concat_Language_year_grp_table.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/research strend in Africa wos/data/languages_table_count_countries_year_group.csv')
data_concat_Language_year_grp_table

In [ ]:
# Language year

In [ ]:
data_concat_Language2 = data_concat_Language[['year','year_group','Language']].set_index(['year_group','year'])
#data_concat_doc_types3 = data_concat_doc_types2['Document Type'].str.split('; ', expand=True)
data_concat_Language2

In [ ]:
# get dummies to have contries on columns
from functools import reduce
dfs = [pd.get_dummies(data_concat_Language2[col]) for col in data_concat_Language2]
d = reduce(lambda x,y : x.add(y, fill_value=0), dfs)

# data cleaning
cols = [c for c in d.columns if c[-1:] != '.']
df=d[cols]
cols = [c for c in df.columns.to_list() if len(c) != 1]
df=df[cols]

dfs= df.replace(0.0, np.nan).reset_index()
#save data
dfs.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/research strend in Africa wos/data/langues_table_count_years.csv')

dfs

In [ ]:
data_concat_Language_year_table = dfs.groupby(['year']).count()
#save data
data_concat_Language_year_table.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/research strend in Africa wos/data/langues_table_count_years.csv')

data_concat_Language_year_table

In [ ]:
data_concat_Language_year_g_table = dfs.groupby(['year_group']).count()
#save data
data_concat_Language_year_g_table.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/research strend in Africa wos/data/langues_table_count_years_group.csv')

data_concat_Language_year_g_table